### Productivity adjustment (HBS hypothesis)

We compute the ratio $GDP(ppp)US/GDP_(ppp)F$ and we multiply it by the Real Exchange Rates Qt, such that:
- Most productive countries will face a lower increase of Q
- Less productive countries will face a higher increase of Q

In [11]:
import pandas as pd

df_GDP = pd.read_csv("/Users/hirecheariles/Documents/Cours/Master Finance/ARP/GDPPPPData.csv", sep=";", skiprows=4)

df_GDP.drop(["Indicator Name", "Indicator Code", "2025", "Unnamed: 70"], axis=1, inplace = True)

df_GDP = df_GDP.loc[:, ["Country Code"]].join(df_GDP.loc[:, "1990":]).set_index("Country Code").T
US_GDP = df_GDP.loc[:, "USA"]
df_GDP = df_GDP.loc[:, ["AUS", "BRA", "CAN", "CHL", "EMU", "HUN", "IND", "IDN", "ISR", "JPN", "MEX", "NZL", "NOR", "POL", "RUS", "SGP", "ZAF", "SWE", "CHE", "CZE", "GBR"]]

GDPmapping = pd.DataFrame(index = ["AUS", "BRA", "CAN", "CHL", "EMU", "HUN", "IND", "IDN", "ISR", "JPN", "MEX", "NZL", "NOR", "POL", "RUS", "SGP", "ZAF", "SWE", "CHE", "CZE", "GBR"],
                       data = ['AUD', 'BRL', 'CAD', 'CLP', 'EUR', 'HUF', "INR", "IDR", "ILS", "JPY", "MXN", "NZD", "NOK", "PLN", "RUB", "SGD", "ZAR", "SEK", "CHF", "CZK", "GBP"]).to_dict()[0]

df_GDP = df_GDP.rename(columns=GDPmapping)
df_GDP = df_GDP.rdiv(US_GDP, axis=0)  #In this strat we scale upward all the real exchange rates, with more magnitude for the less productive countries
df_GDP.index = pd.to_datetime(df_GDP.index, format="%Y")

We import the previous dataframe.

In [ ]:
from data import spt_dol_import
from data import PPP_import
import pandas as pd

spt_dol = spt_dol_import()
df_PPP = PPP_import()

mapping = {'AUSTDOL(ER)': "AUD", 'BRACRUZ(ER)':'BRL', 'CNDOLLR(ER)':'CAD', 'CHILPES(ER)':'CLP',
       'USEURSP(ER)':"EUR", 'HUNFORT(ER)':"HUF", 'INDRUPE(ER)':"INR", 'INDORUP(ER)':"IDR",
       'ISRSHEK(ER)':"ILS", 'JAPAYEN(ER)':"JPY", 'MEXPESO(ER)':"MXN", 'NZDOLLR(ER)':"NZD",
       'NORKRON(ER)':"NOK", 'POLZLOT(ER)':"PLN", 'CISRUBM(ER)':"RUB", 'SINGDOL(ER)':"SGD",
       'COMRAND(ER)':"ZAR", 'SWEKRON(ER)':"SEK", 'SWISSFR(ER)':"CHF", 'CZECHCM(ER)':"CZK", 'GBP(ER)':"GBP"}
spt_dol = spt_dol.rename(columns=mapping)

#The median rebalancement try to avoid any noise from the beginning of a period
rebal_mth = spt_dol.index.to_series().groupby(spt_dol.index.to_period("M")).min() #Monthly rebalancement
rebal_qtr = spt_dol.index.to_series().groupby(spt_dol.index.to_period('Q')).min() #Quarterly rebalancement
rebal_qmed = spt_dol.index.to_series().groupby(spt_dol.index.to_period("Q")).median().dt.normalize() #Quarterly median date

In [60]:
#We reindex the data to get a dataframe with observations at the right intervals
spt_mth = spt_dol.reindex(rebal_mth, method="ffill")
spt_qtr = spt_dol.reindex(rebal_qtr, method ="ffill")
spt_qmed = spt_dol.reindex(rebal_qmed, method="ffill")

df_PPPm = df_PPP.reindex(rebal_mth, method='ffill')
df_PPPq = df_PPP.reindex(rebal_qtr, method="ffill")
df_PPPqmed = df_PPP.reindex(rebal_qmed, method="ffill")

df_GDPm = df_GDP.reindex(rebal_mth, method="ffill")
df_GDPq = df_GDP.reindex(rebal_qtr, method="ffill")
df_GDPqmed = df_GDP.reindex(rebal_qmed, method="ffill")

#Computing the adjusted Real Exchange Rate Qt*GDPus/GDPf for the different periods
df_adjRERm = spt_mth * df_PPPm * df_GDPm
df_adjRERq = spt_qtr * df_PPPq * df_GDPq  #data verified for EUR 2024_01-01
df_adjRERqmed = spt_qmed * df_PPPqmed * df_GDPqmed

In [61]:
#df_adjRERq.rank(axis=1).head()

In [62]:
spt_IS = spt_dol[:"2015"]
spt_OS = spt_dol["2015":]

df_adjRERmIS = df_adjRERm[:"2015"]
df_adjRERmOS = df_adjRERm["2015":]

df_adjRERqIS = df_adjRERq[:"2015"]
df_adjRERqOS = df_adjRERq["2015":]

df_adjRERqmedIS = df_adjRERqmed[:"2015"]
df_adjRERqmedOS = df_adjRERqmed["2015":]

rebal_mIS = rebal_mth[:"2015"]
rebal_qIS = rebal_qtr[:"2015"]
rebal_qmedIS = rebal_qmed[:"2015"]

rebal_mOS = rebal_mth["2015":]
rebal_qOS = rebal_qtr["2015":]
rebal_qmedOS = rebal_qmed["2015":]

In [63]:
import numpy as np
from data import weightsPPPstrat

weights_mIS = weightsPPPstrat(df_adjRERmIS, n_long=7)
weights_qIS = weightsPPPstrat(df_adjRERqIS, n_long=7)
weights_qmedIS = weightsPPPstrat(RERdf = df_adjRERqmedIS, n_long=7, rebal_dates=rebal_qmedIS)

monthly_retIS = np.log(spt_IS) - np.log(spt_IS.shift())
quarter_retIS = np.log(spt_IS.reindex(rebal_qIS)) - np.log(spt_IS.reindex(rebal_qIS).shift()) #Quarterly log returns
qmed_retIS = np.log(spt_IS.reindex(rebal_qmedIS, method="ffill")) - np.log(spt_IS.reindex(rebal_qmedIS, method="ffill").shift(1))

returns_mIS = (weights_mIS * monthly_retIS).sum(axis=1)
returns_qIS = (weights_qIS * quarter_retIS).sum(axis=1)
returns_qmedIS = (weights_qmedIS * qmed_retIS).sum(axis=1)

print(f"SharpeMIS = {12*returns_mIS.mean()/(returns_mIS.std()*np.sqrt(12))}")
print(f"SharpeQIS = {4*returns_qIS.mean()/(returns_qIS.std()*np.sqrt(4))}")
print(f"SharpeQmedIS = {4*returns_qmedIS.mean()/(returns_qmedIS.std()*np.sqrt(4))}")

returns_mIS.describe()

SharpeMIS = -0.04956568938826948
SharpeQIS = -0.5667283408644223
SharpeQmedIS = -0.6217547471973948


count    4696.000000
mean       -0.000013
std         0.000896
min        -0.023473
25%         0.000000
50%         0.000000
75%         0.000000
max         0.010736
dtype: float64

The strategy is even worse within the whole basket... we must include the interest rate differential for the carry return.

### Restriction on the G10 currencies

In [66]:
df_adjRERG10qIS = df_adjRERqIS[["AUD", "CAD", "CHF", "EUR", "GBP", "JPY", "NOK", "NZD", "SEK"]]
df_adjRERG10mIS = df_adjRERmIS[["AUD", "CAD", "CHF", "EUR", "GBP", "JPY", "NOK", "NZD", "SEK"]]

weightsG10qIS = weightsPPPstrat(df_adjRERG10qIS) #Calling the function to copmute weights
weightsG10mIS = weightsPPPstrat(df_adjRERG10mIS)

## Quarterly Strategy
spt_G10qIS = spt_dol.reindex(rebal_qIS, method="ffill") #Only keeping quarters
logret_G10qIS = np.log(spt_G10qIS) - np.log(spt_G10qIS.shift(1))

returns_G10qIS =(logret_G10qIS * weightsG10qIS).sum(axis=1) #Total on each qtr

#Monthly strategy
spt_G10mIS = spt_dol.reindex(rebal_mIS, method="ffill")
logret_G10mIS = np.log(spt_G10mIS) - np.log(spt_G10mIS.shift(1))

returns_G10mIS =(logret_G10mIS * weightsG10mIS).sum(axis=1) 

print(f"Sharpe = {(4*returns_G10qIS.mean()-0.01)/(np.sqrt(4)*returns_G10qIS.std())}")
print(f"Sharpe = {(12*returns_G10mIS.mean()-0.01)/(np.sqrt(12)*returns_G10mIS.std())}")

returns_G10qIS.describe()

Sharpe = -0.05157939747285019
Sharpe = -0.10939799265339838


count    72.000000
mean      0.001671
std       0.032130
min      -0.054940
25%      -0.015089
50%      -0.000874
75%       0.015955
max       0.125707
dtype: float64

### We can get more precisions of the PPP spot rate within a year using CPI differential to modelise intra-year dynamics of prices level